In [2]:
from google.colab import files

uploaded = files.upload()

Saving 등록인구(연령별_동별)_20260922163622.csv to 등록인구(연령별_동별)_20260922163622.csv


In [3]:
import os

print(os.listdir("/content"))

['.config', '등록인구(연령별_동별)_20260922163622.csv', 'sample_data']


In [4]:
import pandas as pd

file_path = "등록인구(연령별_동별)_20260922163622.csv"

raw = pd.read_csv(file_path, encoding="utf-8-sig")

print("원본 shape:", raw.shape)
print()
print(raw.head(10))
print()
print(raw["동별(1)"].unique())
print(raw["항목"].value_counts())

원본 shape: (73, 47)

   동별(1)  동별(2)     항목 2025 3/4 2025 3/4.1 2025 3/4.2 2025 3/4.3 2025 3/4.4  \
0  동별(1)  동별(2)     항목       합계       0~4세       5~9세     10~14세     15~19세   
1     합계     소계      계  9594329     205729     269467     354656     382951   
2     합계     소계    한국인  9318500     201692     263714     349856     370857   
3     합계     소계  등록외국인   275829       4037       5753       4800      12094   
4    강남구     소계      계   562382      12359      19915      32507      31892   
5    강남구     소계    한국인   556324      12302      19840      32435      31736   
6    강남구     소계  등록외국인     6058         57         75         72        156   
7    강남구    신사동      계    15327        334        458        686        767   
8    강남구    신사동    한국인    15114        334        453        680        762   
9    강남구    신사동  등록외국인      213          -          5          6          5   

  2025 3/4.5 2025 3/4.6  ... 2025 4/4.12 2025 4/4.13 2025 4/4.14 2025 4/4.15  \
0     20~24세     25~29세  ...  

In [5]:
df = raw.copy()

# 실제 연령명이 들어 있는 첫 번째 행
age_labels = df.iloc[0]

new_columns = ["district", "dong_name", "population_type"]

for col in df.columns[3:]:
    quarter = "2025Q3" if col.startswith("2025 3/4") else "2025Q4"
    age = age_labels[col]

    if age == "합계":
        age = "total"

    new_columns.append(f"{quarter}_{age}")

df.columns = new_columns

# 연령명이 들어 있던 첫 행 제거
df = df.iloc[1:].reset_index(drop=True)

print(df.shape)
print(df.columns.tolist())

(72, 47)
['district', 'dong_name', 'population_type', '2025Q3_total', '2025Q3_0~4세', '2025Q3_5~9세', '2025Q3_10~14세', '2025Q3_15~19세', '2025Q3_20~24세', '2025Q3_25~29세', '2025Q3_30~34세', '2025Q3_35~39세', '2025Q3_40~44세', '2025Q3_45~49세', '2025Q3_50~54세', '2025Q3_55~59세', '2025Q3_60~64세', '2025Q3_65~69세', '2025Q3_70~74세', '2025Q3_75~79세', '2025Q3_80~84세', '2025Q3_85~89세', '2025Q3_90~94세', '2025Q3_95~99세', '2025Q3_100세 이상', '2025Q4_total', '2025Q4_0~4세', '2025Q4_5~9세', '2025Q4_10~14세', '2025Q4_15~19세', '2025Q4_20~24세', '2025Q4_25~29세', '2025Q4_30~34세', '2025Q4_35~39세', '2025Q4_40~44세', '2025Q4_45~49세', '2025Q4_50~54세', '2025Q4_55~59세', '2025Q4_60~64세', '2025Q4_65~69세', '2025Q4_70~74세', '2025Q4_75~79세', '2025Q4_80~84세', '2025Q4_85~89세', '2025Q4_90~94세', '2025Q4_95~99세', '2025Q4_100세 이상']


In [6]:
# 강남구의 행정동만 추출
# dong_name == '소계'인 강남구 전체 행은 제외
gangnam_pop = df[
    (df["district"] == "강남구") &
    (df["dong_name"] != "소계")
].copy()

# '-' → 0 처리 후 숫자형 변환
value_cols = gangnam_pop.columns[3:]

gangnam_pop[value_cols] = (
    gangnam_pop[value_cols]
    .replace("-", 0)
    .apply(pd.to_numeric, errors="coerce")
)

# 기본 검증
print("shape:", gangnam_pop.shape)
print("행정동 수:", gangnam_pop["dong_name"].nunique())
print()
print("population_type:")
print(gangnam_pop["population_type"].value_counts())
print()
print("행정동 목록:")
print(sorted(gangnam_pop["dong_name"].unique()))
print()
print("결측값:", gangnam_pop.isna().sum().sum())

shape: (66, 47)
행정동 수: 22

population_type:
population_type
계        22
한국인      22
등록외국인    22
Name: count, dtype: int64

행정동 목록:
['개포1동', '개포2동', '개포3동', '개포4동', '논현1동', '논현2동', '대치1동', '대치2동', '대치4동', '도곡1동', '도곡2동', '삼성1동', '삼성2동', '세곡동', '수서동', '신사동', '압구정동', '역삼1동', '역삼2동', '일원1동', '일원본동', '청담동']

결측값: 0


In [7]:
records = []

quarters = ["2025Q3", "2025Q4"]

for dong in gangnam_pop["dong_name"].unique():
    dong_df = gangnam_pop[gangnam_pop["dong_name"] == dong]

    for quarter in quarters:

        total_row = dong_df[dong_df["population_type"] == "계"].iloc[0]
        korean_row = dong_df[dong_df["population_type"] == "한국인"].iloc[0]
        foreigner_row = dong_df[dong_df["population_type"] == "등록외국인"].iloc[0]

        records.append({
            "dong_name": dong,
            "quarter": quarter,
            "resident_total": total_row[f"{quarter}_total"],
            "resident_korean": korean_row[f"{quarter}_total"],
            "resident_foreigner": foreigner_row[f"{quarter}_total"]
        })

resident_base = pd.DataFrame(records)

print("shape:", resident_base.shape)
print()
print(resident_base.head(10))
print()
print("분기별 행 수:")
print(resident_base["quarter"].value_counts())
print()
print("행정동 수:", resident_base["dong_name"].nunique())
print("결측값:", resident_base.isna().sum().sum())

shape: (44, 5)

  dong_name quarter  resident_total  resident_korean  resident_foreigner
0       신사동  2025Q3           15327            15114                 213
1       신사동  2025Q4           15291            15081                 210
2      논현1동  2025Q3           21571            20733                 838
3      논현1동  2025Q4           21500            20647                 853
4      논현2동  2025Q3           20511            20119                 392
5      논현2동  2025Q4           20506            20109                 397
6      압구정동  2025Q3           25671            25465                 206
7      압구정동  2025Q4           25668            25444                 224
8       청담동  2025Q3           24115            23855                 260
9       청담동  2025Q4           25463            25217                 246

분기별 행 수:
quarter
2025Q3    22
2025Q4    22
Name: count, dtype: int64

행정동 수: 22
결측값: 0


In [8]:
resident_base["check"] = (
    resident_base["resident_total"]
    - resident_base["resident_korean"]
    - resident_base["resident_foreigner"]
)

print(resident_base["check"].value_counts())

check
0    44
Name: count, dtype: int64


In [9]:
# 검증용 컬럼 제거
resident_base = resident_base.drop(columns=["check"], errors="ignore")

age_records = []

for dong in gangnam_pop["dong_name"].unique():

    dong_df = gangnam_pop[
        (gangnam_pop["dong_name"] == dong) &
        (gangnam_pop["population_type"] == "계")
    ]

    for quarter in ["2025Q3", "2025Q4"]:

        row = dong_df.iloc[0]

        # SKT 연령구간과 동일하게 구성
        age_10 = (
            row[f"{quarter}_10~14세"] +
            row[f"{quarter}_15~19세"]
        )

        age_20 = (
            row[f"{quarter}_20~24세"] +
            row[f"{quarter}_25~29세"]
        )

        age_30 = (
            row[f"{quarter}_30~34세"] +
            row[f"{quarter}_35~39세"]
        )

        age_40 = (
            row[f"{quarter}_40~44세"] +
            row[f"{quarter}_45~49세"]
        )

        age_50 = (
            row[f"{quarter}_50~54세"] +
            row[f"{quarter}_55~59세"]
        )

        # 60세 이상
        age_60_plus_cols = [
            "60~64세", "65~69세", "70~74세", "75~79세",
            "80~84세", "85~89세", "90~94세",
            "95~99세", "100세 이상"
        ]

        age_60_plus = sum(
            row[f"{quarter}_{col}"]
            for col in age_60_plus_cols
        )

        # 65세 이상
        age_65_plus_cols = [
            "65~69세", "70~74세", "75~79세",
            "80~84세", "85~89세", "90~94세",
            "95~99세", "100세 이상"
        ]

        age_65_plus = sum(
            row[f"{quarter}_{col}"]
            for col in age_65_plus_cols
        )

        age_records.append({
            "dong_name": dong,
            "quarter": quarter,
            "resident_age_10": age_10,
            "resident_age_20": age_20,
            "resident_age_30": age_30,
            "resident_age_40": age_40,
            "resident_age_50": age_50,
            "resident_age_60_plus": age_60_plus,
            "resident_age_65_plus": age_65_plus
        })

resident_age = pd.DataFrame(age_records)

In [10]:
resident_features = resident_base.merge(
    resident_age,
    on=["dong_name", "quarter"],
    how="left"
)

# 외국인 비율
resident_features["foreigner_ratio"] = (
    resident_features["resident_foreigner"]
    / resident_features["resident_total"]
)

# 연령별 비율
for age in ["10", "20", "30", "40", "50", "60_plus", "65_plus"]:
    resident_features[f"resident_age_{age}_ratio"] = (
        resident_features[f"resident_age_{age}"]
        / resident_features["resident_total"]
    )

print("shape:", resident_features.shape)
print("결측값:", resident_features.isna().sum().sum())

print(
    resident_features[
        [
            "dong_name", "quarter",
            "resident_total",
            "resident_age_10_ratio",
            "resident_age_20_ratio",
            "resident_age_30_ratio",
            "resident_age_40_ratio",
            "resident_age_50_ratio",
            "resident_age_60_plus_ratio",
            "resident_age_65_plus_ratio"
        ]
    ].head(10)
)

shape: (44, 20)
결측값: 0
  dong_name quarter  resident_total  resident_age_10_ratio  \
0       신사동  2025Q3           15327               0.094800   
1       신사동  2025Q4           15291               0.096070   
2      논현1동  2025Q3           21571               0.039915   
3      논현1동  2025Q4           21500               0.040465   
4      논현2동  2025Q3           20511               0.059090   
5      논현2동  2025Q4           20506               0.058422   
6      압구정동  2025Q3           25671               0.105216   
7      압구정동  2025Q4           25668               0.106670   
8       청담동  2025Q3           24115               0.090566   
9       청담동  2025Q4           25463               0.090956   

   resident_age_20_ratio  resident_age_30_ratio  resident_age_40_ratio  \
0               0.118940               0.131272               0.157696   
1               0.118174               0.130011               0.157936   
2               0.175096               0.255806               0.170043  

In [11]:
resident_features["age_10plus_ratio_sum"] = (
    resident_features["resident_age_10_ratio"]
    + resident_features["resident_age_20_ratio"]
    + resident_features["resident_age_30_ratio"]
    + resident_features["resident_age_40_ratio"]
    + resident_features["resident_age_50_ratio"]
    + resident_features["resident_age_60_plus_ratio"]
)

print(
    resident_features[
        ["dong_name", "quarter",
         "age_10plus_ratio_sum",
         "resident_age_65_plus_ratio"]
    ].head(10)
)

  dong_name quarter  age_10plus_ratio_sum  resident_age_65_plus_ratio
0       신사동  2025Q3              0.948326                    0.211261
1       신사동  2025Q4              0.948663                    0.211432
2      논현1동  2025Q3              0.976125                    0.149274
3      논현1동  2025Q4              0.976465                    0.150326
4      논현2동  2025Q3              0.960997                    0.163181
5      논현2동  2025Q4              0.961182                    0.165561
6      압구정동  2025Q3              0.933076                    0.215886
7      압구정동  2025Q4              0.934003                    0.216846
8       청담동  2025Q3              0.938503                    0.174663
9       청담동  2025Q4              0.939088                    0.181322


In [12]:
# SKT와 동일하게 10세 이상만을 분모로 사용
resident_features["resident_10plus"] = (
    resident_features["resident_age_10"]
    + resident_features["resident_age_20"]
    + resident_features["resident_age_30"]
    + resident_features["resident_age_40"]
    + resident_features["resident_age_50"]
    + resident_features["resident_age_60_plus"]
)

for age in ["10", "20", "30", "40", "50", "60_plus"]:
    resident_features[f"resident_age_{age}_ratio_10plus"] = (
        resident_features[f"resident_age_{age}"]
        / resident_features["resident_10plus"]
    )

In [13]:
ratio_cols = [
    "resident_age_10_ratio_10plus",
    "resident_age_20_ratio_10plus",
    "resident_age_30_ratio_10plus",
    "resident_age_40_ratio_10plus",
    "resident_age_50_ratio_10plus",
    "resident_age_60_plus_ratio_10plus"
]

resident_features["age_ratio_10plus_check"] = (
    resident_features[ratio_cols].sum(axis=1)
)

print(
    resident_features[
        ["dong_name", "quarter", "age_ratio_10plus_check"]
    ].head(10)
)

  dong_name quarter  age_ratio_10plus_check
0       신사동  2025Q3                     1.0
1       신사동  2025Q4                     1.0
2      논현1동  2025Q3                     1.0
3      논현1동  2025Q4                     1.0
4      논현2동  2025Q3                     1.0
5      논현2동  2025Q4                     1.0
6      압구정동  2025Q3                     1.0
7      압구정동  2025Q4                     1.0
8       청담동  2025Q3                     1.0
9       청담동  2025Q4                     1.0


In [14]:
final_cols = [
    # Key
    "dong_name",
    "quarter",

    # 기본 인구
    "resident_total",
    "resident_korean",
    "resident_foreigner",
    "foreigner_ratio",

    # 연령별 인구수
    "resident_age_10",
    "resident_age_20",
    "resident_age_30",
    "resident_age_40",
    "resident_age_50",
    "resident_age_60_plus",
    "resident_age_65_plus",

    # 전체 등록인구 기준 비율
    "resident_age_10_ratio",
    "resident_age_20_ratio",
    "resident_age_30_ratio",
    "resident_age_40_ratio",
    "resident_age_50_ratio",
    "resident_age_60_plus_ratio",
    "resident_age_65_plus_ratio",

    # SKT 비교를 위한 10세 이상 기준
    "resident_10plus",
    "resident_age_10_ratio_10plus",
    "resident_age_20_ratio_10plus",
    "resident_age_30_ratio_10plus",
    "resident_age_40_ratio_10plus",
    "resident_age_50_ratio_10plus",
    "resident_age_60_plus_ratio_10plus"
]

resident_final = resident_features[final_cols].copy()

print("최종 shape:", resident_final.shape)
print("결측값:", resident_final.isna().sum().sum())
print("중복 Key:",
      resident_final.duplicated(["dong_name", "quarter"]).sum())

print()
print(resident_final.head())

최종 shape: (44, 27)
결측값: 0
중복 Key: 0

  dong_name quarter  resident_total  resident_korean  resident_foreigner  \
0       신사동  2025Q3           15327            15114                 213   
1       신사동  2025Q4           15291            15081                 210   
2      논현1동  2025Q3           21571            20733                 838   
3      논현1동  2025Q4           21500            20647                 853   
4      논현2동  2025Q3           20511            20119                 392   

   foreigner_ratio  resident_age_10  resident_age_20  resident_age_30  \
0         0.013897             1453             1823             2012   
1         0.013734             1469             1807             1988   
2         0.038848              861             3777             5518   
3         0.039674              870             3705             5471   
4         0.019112             1212             2982             4243   

   resident_age_40  ...  resident_age_50_ratio  resident_age_60_plu

In [15]:
import numpy as np

# ① 전체 = 한국인 + 등록외국인
check_population = np.allclose(
    resident_final["resident_total"],
    resident_final["resident_korean"]
    + resident_final["resident_foreigner"]
)

# ② 10세 이상 연령별 인구수 합 = resident_10plus
age_count_cols = [
    "resident_age_10",
    "resident_age_20",
    "resident_age_30",
    "resident_age_40",
    "resident_age_50",
    "resident_age_60_plus"
]

check_age_count = np.allclose(
    resident_final[age_count_cols].sum(axis=1),
    resident_final["resident_10plus"]
)

# ③ SKT 비교용 연령비율 합 = 1
age_ratio_cols = [
    "resident_age_10_ratio_10plus",
    "resident_age_20_ratio_10plus",
    "resident_age_30_ratio_10plus",
    "resident_age_40_ratio_10plus",
    "resident_age_50_ratio_10plus",
    "resident_age_60_plus_ratio_10plus"
]

check_age_ratio = np.allclose(
    resident_final[age_ratio_cols].sum(axis=1),
    1.0
)

print("전체인구 검증:", check_population)
print("연령 인구수 검증:", check_age_count)
print("10세 이상 연령비율 검증:", check_age_ratio)

전체인구 검증: True
연령 인구수 검증: True
10세 이상 연령비율 검증: True


In [16]:
output_file = "gangnam_resident_population_features_2025Q3_Q4.csv"

resident_final.to_csv(
    output_file,
    index=False,
    encoding="utf-8-sig"
)

print("저장 완료:", output_file)

저장 완료: gangnam_resident_population_features_2025Q3_Q4.csv


In [17]:
from google.colab import files

files.download("gangnam_resident_population_features_2025Q3_Q4.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>